# Fine-tune VideoMAE for action recognition

**Foundation model** · maps to lessons C3 (video backbones) and C9 (reproduction).

VideoMAE is a self-supervised video transformer. We fine-tune the pretrained `videomae-base` on a small **UCF101 subset** with the 🤗 `transformers` Trainer — the same recipe transfers directly to egocentric datasets (EPIC-Kitchens, Ego4D).

> **This is the heaviest lab — Runtime → T4 GPU is required** (~20–30 min). It is authored to the official HuggingFace recipe and is meant to be run on Colab; it is not pre-executed here. If an install resolves to an incompatible version, pin as noted in the last cell.

In [ ]:
!pip -q install "transformers>=4.41" "evaluate" "pytorchvideo" "av"
import torch, pytorchvideo, evaluate, numpy as np
from huggingface_hub import hf_hub_download
import os, tarfile
print("GPU:", torch.cuda.is_available())

## 1 · Data — a small UCF101 subset (10 classes)

In [ ]:
archive = hf_hub_download(repo_id="sayakpaul/ucf101-subset", filename="UCF101_subset.tar.gz", repo_type="dataset")
if not os.path.exists("UCF101_subset"):
    with tarfile.open(archive) as t: t.extractall(".")
root = "UCF101_subset"
labels = sorted({p.split("_")[1] for p in os.listdir(f"{root}/train")})
label2id = {l: i for i, l in enumerate(labels)}; id2label = {i: l for l, i in label2id.items()}
print(len(labels), "classes:", labels)

## 2 · Model + processor (pretrained VideoMAE)

In [ ]:
from transformers import VideoMAEImageProcessor, VideoMAEForVideoClassification
ckpt = "MCG-NJU/videomae-base"
processor = VideoMAEImageProcessor.from_pretrained(ckpt)
model = VideoMAEForVideoClassification.from_pretrained(
    ckpt, label2id=label2id, id2label=id2label, ignore_mismatched_sizes=True)
num_frames = model.config.num_frames

## 3 · Frame sampling + augmentation (pytorchvideo)

In [ ]:
from pytorchvideo.data import Ucf101, make_clip_sampler
from pytorchvideo.transforms import ApplyTransformToKey, Normalize, UniformTemporalSubsample
from torchvision.transforms import Compose, Lambda, Resize, RandomCrop, CenterCrop
import pytorchvideo.data

mean, std = processor.image_mean, processor.image_std
size = processor.size["shortest_edge"] if "shortest_edge" in processor.size else processor.size["height"]
clip_dur = num_frames / 25

def tf(train):
    crop = RandomCrop(size) if train else CenterCrop(size)
    return ApplyTransformToKey("video", Compose([
        UniformTemporalSubsample(num_frames), Lambda(lambda x: x / 255.0),
        Normalize(mean, std), Resize(size), crop]))

def ds(split, train):
    return pytorchvideo.data.Ucf101(
        data_path=os.path.join(root, split),
        clip_sampler=make_clip_sampler("random" if train else "uniform", clip_dur),
        decode_audio=False, transform=tf(train))

train_ds, val_ds = ds("train", True), ds("val", False)

## 4 · Train with the 🤗 Trainer

In [ ]:
from transformers import TrainingArguments, Trainer
metric = evaluate.load("accuracy")
def collate(ex):
    return {"pixel_values": torch.stack([e["video"].permute(1,0,2,3) for e in ex]),
            "labels": torch.tensor([label2id[e["label"]] for e in ex])}
def metrics(p):
    return metric.compute(predictions=np.argmax(p.predictions, 1), references=p.label_ids)

args = TrainingArguments("videomae-ucf101", per_device_train_batch_size=4, per_device_eval_batch_size=4,
    learning_rate=5e-5, max_steps=300, eval_strategy="steps", eval_steps=100, logging_steps=20,
    warmup_ratio=0.1, fp16=torch.cuda.is_available(), remove_unused_columns=False, report_to="none")
trainer = Trainer(model, args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=collate, compute_metrics=metrics)
trainer.train()

## 5 · Evaluate & predict one clip

In [ ]:
print(trainer.evaluate())
import itertools
batch = collate(list(itertools.islice(iter(val_ds), 4)))
with torch.no_grad():
    logits = model(batch["pixel_values"].to(model.device)).logits
for pred, gt in zip(logits.argmax(-1).tolist(), batch["labels"].tolist()):
    print("pred:", id2label[pred], "| true:", id2label[gt])

## Save & persist
The 🤗 Trainer already writes checkpoints, `trainer_state.json` (the full loss/eval history) and logs to its `output_dir` ("videomae-ucf101"). Also call `trainer.save_model("videomae-final")` (and optionally `trainer.push_to_hub()`). Colab is ephemeral, so zip + download the folder or mount Google Drive to keep it.

### Notes & next steps
- **Version pins** if something breaks: `transformers==4.44.2`, `pytorchvideo==0.1.5`, `av==12.3.0`.
- Raise `max_steps` and add more classes for real accuracy.
- Point `data_path` at **EPIC-Kitchens-100** / **Ego4D** clips to fine-tune for *egocentric* action recognition (lesson C9). Compare against the frozen-feature baseline from lesson C9's notebook.